# Curriculum 03 · Lab 1 — FAISS: the in-memory vector index

**Goal:** Build a FAISS vector store over a deterministic subset of
rag-mini-wikipedia and inspect the three things that decide how a vector
database behaves: the index type, the score convention, and the persistence
model. FAISS is the workhorse of the *in-memory* end of the vector-store
spectrum — the index lives entirely in RAM, builds in seconds, and searches
exactly.

```
Index       : FAISS IndexFlatL2 — brute-force exact search (no training, no params)
Score       : SQUARED L2 distance — LOWER = more similar (0.0 = perfect match)
Persistence : none — in-RAM; the index vanishes at process exit
Also runs   : MMR (max_marginal_relevance_search) — similarity vs diversity
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Data        : rag-mini-wikipedia (first 100 passages, test questions 1606/1610/1604)
```

**Why FAISS:** it is the honest baseline every other store is measured
against. `IndexFlatL2` compares the query vector to *every* stored vector —
exact search, no approximation — so the ranking you see here is ground
truth. Later labs reuse these exact scores: Chroma (lab 02) must reproduce
them from disk, Qdrant (lab 03) must reproduce them as cosines, and lab 04
benchmarks all three on top.

This is the first lab of track 03-vector-databases (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
imports pandas plus the repo's vector-store classes and BGE embedder, and
puts the repo-root component library on `sys.path` so this notebook reuses
`vectordb/*.py` and `embeddings/bge.py` exactly like the lab script.

**WHY:** Everything embeds **locally** with BGE via sentence-transformers —
no API embeddings anywhere. The store classes live in the repo's shared
component library (`vectordb/`), not inside the lab, so the exact same code
path runs here, in the `.py`, and in later tracks.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script) or from
the notebook's own folder (the Jupyter default) — and `cd`s into it so every
path stays repo-relative.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model is loaded lazily
when the experiment cell first calls it.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings
#   faiss-cpu             -> the FAISS index (IndexFlatL2)
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from embeddings.bge import BGEEmbedding  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** The module-level constants that define the experiment: which corpus
files to read, how many passages to embed, which test questions to ask, how
many hits to print, the MMR settings, and the expected embedding dimension.

**WHY:** `N_PASSAGES = 100` takes the deterministic head of the 3200-passage
corpus so CPU embedding time stays low; `QUESTION_IDS = [1606, 1610, 1604]`
are real rows of `test.parquet` whose answers live inside those 100
passages. `LAMBDA_MULT = 0.5` is MMR's relevance-vs-diversity dial — 1.0 is
pure similarity, 0.0 is pure diversity.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1604]  # real questions from test.parquet, answers inside the subset
TOP_K = 3
MMR_K = 5
LAMBDA_MULT = 0.5  # MMR: 1.0 = pure similarity, 0.0 = pure diversity
PREVIEW = 62  # max characters of passage text shown next to each hit
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DIM = 768
FLOAT_BYTES = 4  # float32


## 2 · Load — corpus + questions from the fresh parquet files

**WHAT:** Three small readers: `load_passages` pulls the first `n` passages
(text + ids) from `passages.parquet`; `load_questions` pulls specific rows by
id from `test.parquet` and returns `(question_id, question_text)` pairs;
`preview` flattens a passage for one-line printing and `passage_lookup` maps
hit ids back to printable text.

**WHY:** The corpus lives in the repo's fresh rag-mini-wikipedia set
(`Data/corpus/rag-mini-wikipedia/`), manifest-verified and fully local — no
downloads, no randomness. The ids let the retrieval section label every hit
back to a specific passage, which is how we see *where* each store finds an
answer.


In [4]:
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def passage_lookup(texts: list[str], ids: list[int]) -> dict[int, str]:
    """Map passage id -> text, for turning a hit id back into printable text."""
    return dict(zip(ids, texts))


## 3 · Experiment — embed, index, query, diversify

**WHAT:** `run_experiment` is the whole lab in one function: it embeds the
100 passages in one batched call and each query individually (`embed_query`
— exactly how real RAG behaves), builds the FAISS store, scores every
question, and runs one MMR retrieval. It returns a dict of artifacts instead
of printing, so the demo and the verification gate read the same numbers
without recomputing anything.

**WHY:** One embed + one index, consumed twice, is the anti-pattern guard
this track uses everywhere: the demo you read and the gate that verifies
must look at the *same* run, or a discrepancy could slip through unnoticed.


In [5]:
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed the whole subset once (batched) + each question once ---------
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME)
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    question_texts = [qtext for _, qtext in questions]
    query_vecs = [embedder.embed_query(q) for q in question_texts]

    # --- Build the FAISS index (in-memory) ----------------------------------
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    store = FAISSVectorStore()
    t0 = time.perf_counter()
    store.add(chunks, embeddings=passage_vecs)
    index_s = time.perf_counter() - t0

    # --- Query each question with scores ------------------------------------
    scored = [
        store.query_with_scores(qvec, top_k=TOP_K) for qvec in query_vecs
    ]

    # --- MMR on the first question ------------------------------------------
    mmr_docs = store.query_mmr(query_vecs[0], top_k=MMR_K, lambda_mult=LAMBDA_MULT)

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "query_vecs": query_vecs,
        "embed_s": embed_s,
        "index_s": index_s,
        "scored": scored,
        "mmr_docs": mmr_docs,
        "dim": len(passage_vecs[0]),
        "indexed": len(passage_vecs),
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — one embed, one index build per store,
all queries — and keeps the artifact dict as `exp`.

**WHY:** Everything after this cell (the demo and the verification gate)
reads from this single `exp`, so the printed numbers and the verified
numbers are guaranteed to come from the same run.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the lab's story: the deterministic corpus, the
embed + index timings (plus the in-memory size in KB — `100 x 768 x 4 bytes`
of float32), the top-3 per question as squared-L2 distances, and one MMR run.

**WHY:** Read the scores like a retrieval practitioner: lower squared-L2 =
more similar, and the ordering is what matters — which passage lands on top,
and how far the others fall. The MMR block shows the same candidates
re-ranked for diversity, which is why its order differs from the pure
similarity ranking above.


In [7]:
def print_demo(exp: dict) -> None:
    passage_lk = passage_lookup(exp["passage_texts"], exp["passage_ids"])
    n_bytes = exp["indexed"] * exp["dim"] * FLOAT_BYTES

    print("=" * 66)
    print("Lab 01 — FAISS: the in-memory vector index")
    print(f"{BGE_MODEL_NAME} | exact flat-L2 index | in-memory only")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    {len(exp['questions'])} questions from test.parquet:")
    for qid, qtext in exp["questions"]:
        print(f"      [{qid}] {qtext}")

    print(f"\n[2] Embed + index:")
    print(f"    embedded {exp['indexed']} passages in {exp['embed_s']:.2f}s (dim {exp['dim']})")
    print(f"    FAISS index built in {exp['index_s']:.3f}s")
    print(f"    in-memory size ~ {n_bytes / 1024:.0f} KB "
          f"({exp['indexed']} x {exp['dim']} x {FLOAT_BYTES}B float32)")

    print(f"\n[3] Top-{TOP_K} per question (score = squared L2 distance, LOWER = more similar):")
    for i, (qid, qtext) in enumerate(exp["questions"]):
        print(f'\n    Q[{qid}] "{qtext}"')
        for doc, score in exp["scored"][i]:
            pid = doc.metadata.get("id", "?")
            print(f"      {score:8.4f}  [passage {pid}] {preview(doc.page_content)}")
        if i == 0:
            print("      ^ note: 0.0 would be a perfect match; these distances grow")
            print("        as relevance drops")

    print(f"\n[4] MMR on Q[{exp['questions'][0][0]}] (lambda_mult={LAMBDA_MULT}, k={MMR_K}):")
    for rank, doc in enumerate(exp["mmr_docs"], 1):
        pid = doc.metadata.get("id", "?")
        print(f"      {rank}. [passage {pid}] {preview(doc.page_content)}")
    print("      MMR re-ranks the candidates: raise lambda_mult toward 1.0 for")
    print("      pure relevance, lower it toward 0.0 for pure diversity.")

    print("\n[5] Takeaway")
    print("    FAISS's default flat-L2 index is exact, in-RAM, and reports")
    print("    squared-L2 distances (lower = better). It is the fastest store")
    print("    to build and the honest baseline for lab 04's benchmark — but")
    print("    everything vanishes at process exit. Chroma (lab 02) trades a")
    print("    few milliseconds for an on-disk store; Qdrant (lab 03) trades")
    print("    them for a cosine score and a full query language.")


In [8]:
print_demo(exp)


Lab 01 — FAISS: the in-memory vector index
BAAI/bge-base-en-v1.5 | exact flat-L2 index | in-memory only

[1] Corpus (deterministic subset, no randomness):
    100 passages (first 100 of 3200, ids 0..99)
    3 questions from test.parquet:
      [1606] Is Uruguay's capital Montevideo?
      [1610] Who founded Montevideo?
      [1604] Is Uruguay located in the northwesten part of Africa?

[2] Embed + index:
    embedded 100 passages in 15.37s (dim 768)
    FAISS index built in 0.049s
    in-memory size ~ 300 KB (100 x 768 x 4B float32)

[3] Top-3 per question (score = squared L2 distance, LOWER = more similar):

    Q[1606] "Is Uruguay's capital Montevideo?"
        0.2650  [passage 36] Montevideo, Uruguay's capital.
        0.4931  [passage 15] Uruguay's capital, Montevideo, was founded by the Spanish in t...
        0.5141  [passage 64] Montevideo, capital of the country. A view of pedestrian stree...
      ^ note: 0.0 would be a perfect match; these distances grow
        as relevance 

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate` the lab script runs with `--verify`:
dimension and count, one `TOP_K`-sized hit list per question, squared-L2
scores ascending with rank, the two content checks (Q1610's top-1 names the
Spanish founder of Montevideo, Q1606's mentions Montevideo), and MMR
returning `MMR_K` distinct documents.

**WHY:** The gate is the lab's contract — `python 01-faiss.py --verify`
must print 7/7 PASS, and this cell proves the notebook reproduces the
verified `.py` exactly.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Dimension and count match the model / subset.
    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == BGE_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Every question returned exactly TOP_K scored hits.
    checks.append(
        ("each question returns TOP_K scored hits",
         all(len(hits) == TOP_K for hits in exp["scored"]))
    )

    # Squared-L2 scores ascend with rank (0.0 would be a perfect match).
    scores_ascending = all(
        [s for _, s in hits] == sorted(s for _, s in hits) for hits in exp["scored"]
    )
    checks.append(("squared-L2 scores ascend per query (lower = more similar)", scores_ascending))

    # Content check: Q1610 "Who founded Montevideo?" must rank the passage
    # that says the Spanish founded Montevideo at #1 (passage id 2 lives
    # inside the first N_PASSAGES).
    q1610_top = exp["scored"][1][0][0].page_content.lower()
    checks.append(("Q1610 top-1 names the Spanish founder of Montevideo", "spanish" in q1610_top))

    # Q1606 "Is Uruguay's capital Montevideo?" must rank an Uruguay passage
    # that mentions Montevideo at #1.
    q1606_top = exp["scored"][0][0][0].page_content.lower()
    checks.append(("Q1606 top-1 mentions Montevideo", "montevideo" in q1606_top))

    # MMR returns exactly MMR_K distinct documents (no duplicates).
    mmr_texts = [d.page_content for d in exp["mmr_docs"]]
    checks.append(("MMR returns MMR_K distinct documents", len(mmr_texts) == MMR_K and len(set(mmr_texts)) == MMR_K))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


if __name__ == "__main__":
    exp = run_experiment()
    if "--verify" in sys.argv:
        sys.exit(verify_gate(exp))
    print_demo(exp)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Lab 01 — FAISS: the in-memory vector index
BAAI/bge-base-en-v1.5 | exact flat-L2 index | in-memory only

[1] Corpus (deterministic subset, no randomness):
    100 passages (first 100 of 3200, ids 0..99)
    3 questions from test.parquet:
      [1606] Is Uruguay's capital Montevideo?
      [1610] Who founded Montevideo?
      [1604] Is Uruguay located in the northwesten part of Africa?

[2] Embed + index:
    embedded 100 passages in 6.31s (dim 768)
    FAISS index built in 0.003s
    in-memory size ~ 300 KB (100 x 768 x 4B float32)

[3] Top-3 per question (score = squared L2 distance, LOWER = more similar):

    Q[1606] "Is Uruguay's capital Montevideo?"
        0.2650  [passage 36] Montevideo, Uruguay's capital.
        0.4931  [passage 15] Uruguay's capital, Montevideo, was founded by the Spanish in t...
        0.5141  [passage 64] Montevideo, capital of the country. A view of pedestrian stree...
      ^ note: 0.0 would be a perfect match; these distances grow
        as relevance d

In [10]:
verify_gate(exp)


verification gate:
  [PASS] embedding dimension is 768 (BGE base)
  [PASS] exactly 100 passages indexed
  [PASS] each question returns TOP_K scored hits
  [PASS] squared-L2 scores ascend per query (lower = more similar)
  [PASS] Q1610 top-1 names the Spanish founder of Montevideo
  [PASS] Q1606 top-1 mentions Montevideo
  [PASS] MMR returns MMR_K distinct documents


0